In [1]:
# Lab 7
# Model assessment

In [23]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io

import glob
from PIL import Image

import utils
import platform
print("Torch Loaded")

Torch Loaded


In [24]:
def get_platform_device():
    '''

    Detects the OS and the best GPU to use on the device

    '''
    
    os_name = platform.system()
    processor = platform.processor()
    

    if torch.cuda.is_available():
        device = torch.device("cuda")
        platform_info = f"Nvidia GPU via CUDA ({os_name})"
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        platform_info = f"Apple Silicon via MPS ({os_name})"
    else:
        device = torch.device("cpu")
        platform_info = f"CPU fallback ({os_name} {processor})"
    
    print(f"Platform detected: {platform_info}")
    return device

device = get_platform_device()
print(f"Using device: {device}")

Platform detected: Apple Silicon via MPS (Darwin)
Using device: mps


In [25]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

In [26]:
vgg_model.requires_grad_(False)
next(iter(vgg_model.parameters())).requires_grad

False

In [27]:
vgg_model.classifier[0:3]

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
)

In [28]:
N_CLASSES = 6

my_model = nn.Sequential(
    vgg_model.features,
    vgg_model.avgpool,
    nn.Flatten(),
    vgg_model.classifier[0:3],
    nn.Linear(4096, 500),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(500, N_CLASSES)
)

loss_function = nn.CrossEntropyLoss()
optimizer = Adam(my_model.parameters(), lr=1e-4)
backend = "aot_eager" if device.type == "mps" else "inductor"   # Inductor breaks sometimes
model = torch.compile(my_model.to(device), backend=backend)
print(f"Using backend: {backend}")

Using backend: aot_eager


In [29]:
pre_trans = weights.transforms()

In [30]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.RandomRotation(25),
    transforms.RandomHorizontalFlip(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [31]:
DATA_LABELS = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"] 
    
class MyDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.paths = []
        self.labels = []
        self.transform = transform
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = glob.glob(data_dir + label + '/*.png')
            for path in data_paths:
                self.paths.append(path)
                self.labels.append(l_idx)

    def __getitem__(self, idx):
        img = tv_io.read_image(self.paths[idx], tv_io.ImageReadMode.RGB).float() / 255.0
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx])

    def __len__(self):
        return len(self.paths)

In [32]:
train_path = "data/fruits/train/"
train_data = MyDataset(train_path, transform=train_transform) 
train_loader = DataLoader(train_data, batch_size=n, shuffle=True)
train_N = len(train_loader.dataset)

valid_path = "data/fruits/valid/"
valid_data = MyDataset(valid_path, transform=train_transform)
valid_loader = DataLoader(valid_data, batch_size=n, shuffle=False)
valid_N = len(valid_loader.dataset)

In [33]:
def get_batch_accuracy(output, y):
    _, pred = torch.max(output, dim=1)
    return (pred == y).float().mean().item()

In [34]:
def train(model, loader):
    model.train()
    running_loss, running_acc = 0.0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = loss_function(output, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        running_acc += get_batch_accuracy(output, y) 
    return running_loss / len(loader), running_acc / len(loader)

In [35]:
def validate(model, loader): # FIXED: Added loader to arguments
    model.eval()
    running_loss, running_acc = 0.0, 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = loss_function(output, y)
            running_loss += loss.item()
            running_acc += get_batch_accuracy(output, y) 
    return running_loss / len(loader), running_acc / len(loader)

In [36]:
def train_model(epochs: int) -> None:
    for epoch in range(epochs):
        t_loss, t_acc = train(model, train_loader)
        v_loss, v_acc = validate(model, valid_loader)
        print(f"Epoch: {epoch+1} | Train - Loss: {t_loss:.4f} Accuracy: {t_acc:.4f} | Valid - Loss: {v_loss:.4f} Accuracy: {v_acc:.4f}")

In [37]:
train_model(10)

Epoch: 1 | Train - Loss: 1.0495 Accuracy: 0.6612 | Valid - Loss: 0.4733 Accuracy: 0.8785
Epoch: 2 | Train - Loss: 0.3826 Accuracy: 0.8901 | Valid - Loss: 0.2466 Accuracy: 0.9403
Epoch: 3 | Train - Loss: 0.2572 Accuracy: 0.9256 | Valid - Loss: 0.1775 Accuracy: 0.9574
Epoch: 4 | Train - Loss: 0.2152 Accuracy: 0.9255 | Valid - Loss: 0.1488 Accuracy: 0.9631
Epoch: 5 | Train - Loss: 0.1729 Accuracy: 0.9476 | Valid - Loss: 0.1691 Accuracy: 0.9331
Epoch: 6 | Train - Loss: 0.1377 Accuracy: 0.9568 | Valid - Loss: 0.1332 Accuracy: 0.9631
Epoch: 7 | Train - Loss: 0.1333 Accuracy: 0.9568 | Valid - Loss: 0.1172 Accuracy: 0.9602
Epoch: 8 | Train - Loss: 0.1298 Accuracy: 0.9620 | Valid - Loss: 0.1244 Accuracy: 0.9631
Epoch: 9 | Train - Loss: 0.1310 Accuracy: 0.9567 | Valid - Loss: 0.1241 Accuracy: 0.9574
Epoch: 10 | Train - Loss: 0.1114 Accuracy: 0.9602 | Valid - Loss: 0.1293 Accuracy: 0.9602


In [38]:
vgg_model.requires_grad_(True)
optimizer = Adam(my_model.parameters(), lr=.0001)


In [39]:
train_model(1)

Epoch: 1 | Train - Loss: 0.1071 Accuracy: 0.9619 | Valid - Loss: 0.1020 Accuracy: 0.9688


In [ ]:
# Passed: Accuracy - 96%